In [2]:
import warnings
warnings.filterwarnings('ignore')

from langchain_community.document_loaders import PyPDFLoader
loader = PyPDFLoader('GK_Questions.pdf')
pages = loader.load()

from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=30)
chunk_fun = splitter.split_documents(pages)
chunk = [i.page_content for i in chunk_fun]
metadata = [i.metadata for i in chunk_fun]
metadata[0]

{'producer': 'ReportLab PDF Library - (opensource)',
 'creator': '(unspecified)',
 'creationdate': '2026-07-09T11:07:34+00:00',
 'author': '(anonymous)',
 'keywords': '',
 'moddate': '2026-07-09T11:07:34+00:00',
 'subject': '(unspecified)',
 'title': '(anonymous)',
 'trapped': '/False',
 'source': 'GK_Questions.pdf',
 'total_pages': 19,
 'page': 0,
 'page_label': '1'}

In [3]:
import chromadb
from chromadb.utils.embedding_functions import SentenceTransformerEmbeddingFunction
embedding = SentenceTransformerEmbeddingFunction()

client = chromadb.PersistentClient(path='./multi_agentic_rag_3')
collection = client.get_or_create_collection(name='Collection', embedding_function=embedding)

if collection.count() == 0:
    collection.add(
        documents=chunk,
        ids = [str(i) for i in range(len(chunk))],
        metadatas= metadata
    )
collection.count()    

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2955.94it/s]


37

In [4]:
from langchain_groq import ChatGroq
llm = ChatGroq(model='openai/gpt-oss-120b')
import os
from dotenv import load_dotenv
load_dotenv()
key = os.getenv('GROQ_API_KEY')

import ast
from langchain_core.tools import tool
import ast
from langchain_core.tools import tool

@tool
def calculator(expression:str):
    '''This  tool is useful for arithmetic operations'''
    try:
        response = ast.literal_eval(expression)
        return str(response)
    except Exception as e:
        return str(e)


In [5]:
@tool
def retrive(query:str):
    '''Do evry userasking fine the local document'''
    prompt = 'Do the query simetic search{query}'
    query_rewrite = llm.invoke(prompt)
    query_rewrite = query_rewrite.content()

    result = collection.query(query_texts=[query_rewrite], n_results=8)
    distence = result['distances'][0]
    document = result['documents'][0]
    threshold = 1.0
    good_chunk = []
    print(distence)
    for doc, dis in zip(distence, document):
        if dis < threshold:
            good_chunk.append(doc)
        if not good_chunk:
            return 'THIS NOT MY CONTENT'
        return '\n\n'.join (good_chunk)    

In [6]:
tools_atribute = [calculator, retrive]
tool_name = {t.name : t for t in tools_atribute}
tool_name

{'calculator': StructuredTool(name='calculator', description='This  tool is useful for arithmetic operations', args_schema=<class 'langchain_core.utils.pydantic.calculator'>, func=<function calculator at 0x00000273A026C4A0>),
 'retrive': StructuredTool(name='retrive', description='Do evry userasking fine the local document', args_schema=<class 'langchain_core.utils.pydantic.retrive'>, func=<function retrive at 0x00000273A0478360>)}

In [9]:
calculator_agent = llm.bind_tools([calculator])
retrive_agent = llm.bind_tools([retrive])

agent = {
    'calculator_agent':calculator_agent,
    'retrive_agent':retrive_agent
}

In [10]:
def find_agent(query):
    prompt = f'''
    Choose one tool for one work.

    If there are arithmetic operations, use calculator_agent.
    Every other question should use retrive_agent

    question = {query}

    return only one of:
    calculator_agent,
    retrive_agent
    '''

    response = llm.invoke(prompt)
    label = response.content.lower().strip()

    if label == 'calculator_agent':
        return 'calculator_agent'
    elif label == 'retrive_agent':
        return 'retrive_agent'
    else:
        'general'
        

In [15]:
def search(query:str):
    label = find_agent(query=query)

    if label == "general":
        response = llm.invoke(label)
        return response.content 
    agent_search = agent[label]
    response = agent_search.invoke(query)

    if not response.tool_calls:
        return response.content
    for call in response.tool_calls:
        args = call['args']
        name = call['name']

        result = tool_name[name].invoke(args)
    final_result = llm.invoke(result)
    return final_result.content    



In [21]:
q = 'how is happen 143*2'
response = search(query=q)
print(response)

The product of \(143 \times 2\) is:

\[
143 \times 2 = 286
\]

So the result is **286**. If you’d like to see how the calculation works step‑by‑step:

1. Multiply the units digit: \(3 \times 2 = 6\).  
2. Multiply the tens digit: \(4 \times 2 = 8\).  
3. Multiply the hundreds digit: \(1 \times 2 = 2\).

Putting those together gives **286**.
